# Session 3 Lab — One Growth Model, Every Algorithm

This lab runs Deck 3.5, *The Growth Model Across Algorithms*, page by page. It solves one stochastic growth model
with every method of the session, first **with the model in hand**, then **without it**, and ends with the deck's
two side-by-side tables, filled in with measured numbers. Every learner is graded against the **exact
dynamic-programming solution**, in units of consumption.

| Block | Deck 3.5 | You will run | Code |
|---|---|---|---|
| **A** | One Model, One Interface; VFI and PI | `reset`/`step`/`branches`, the exact answer by value function iteration and by policy iteration, the grade | book ch. 11–12; `session03_extra.py` |
| **B** | the simplest actor–critic and its variants | eight numbers on **continuous** capital, the branches summed; then one drawn branch, the tree, the walk | `session03_demo.py` |
| **C** | deep value iteration | an actor and a critic network on Session 2's deterministic model; one step or many; Euler or Bellman | `session03_value.py` |
| **D** | Monte Carlo evaluation and control | whole simulated paths | book ch. 11 |
| **E** | TD(0), n-step, TD(λ); SARSA, expected SARSA; Q-learning, double Q | one step at a time | book ch. 12; `session03_extra.py` |
| **F** | linear TD and LSTD; DQN | weights and a network in place of the table | book ch. 12 |
| **G** | REINFORCE; actor–critic without the model | the rule moved by the score; on the grid, then the eight numbers on continuous capital | book ch. 12; `session03_demo.py` |
| **H** | With the Model / Without the Model, Side by Side | every row of the two tables, measured; then a continuous action (a portfolio share) | — |

Blocks A–C read the model; blocks D–G see only `step`, through a wrapper that refuses anything else. The book's
code — Miao & Feng, *AI for Economists*, chapters 11 and 12 — is vendored unchanged in `session03_assets/book_code/`
(two import-path fixes, listed in `MANIFEST.json`); the rows the book does not ship (policy iteration, TD prediction
with n-step and λ, expected SARSA, double Q) are in the course-owned `session03_extra.py`. The notebook runs
offline on a CPU from a clean kernel in about ten minutes, because each block uses a **reduced budget**; the
full-budget results the slides quote are in `book_recorded_results.json`. Each block ends with a **gate**: a
bound fixed before the run, from a five-seed pilot.

**Calibration.** The grid (blocks A, D–G) and the continuous model of blocks B and G share the book's numbers:
α = 0.36, β = 0.95, δ = 0.10, u = ln c, z ∈ {0.9, 1.1} with switching probability 0.2. **Block C uses Session 2's
deterministic model** (α = 0.33; rung 0: β = 0.96, δ = 1, log utility; rung 1: β = 0.99, δ = 0.025, CRRA 2).
Losses are compared only within one model.

In [ ]:
%matplotlib inline
import json, math, sys, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

LAB = Path.cwd() if (Path.cwd() / "session03_assets").is_dir() else Path.cwd() / "labs"
sys.path.insert(0, str(LAB / "session03_assets"))
import session03_env as s3            # puts the book's ch. 11 and ch. 12 code on the path
import session03_value as value       # block C
import session03_extra as x           # policy iteration, TD prediction, expected SARSA, double Q

OUT = LAB / "outputs" / "session03"
OUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
RECORDED = json.loads((LAB / "session03_assets" / "book_recorded_results.json").read_text(encoding="utf-8"))
print("torch", torch.__version__, "| numpy", np.__version__, "| device: cpu")
print("recorded full-budget runs:", ", ".join(RECORDED["scripts"]))

# Every block adds its rows here; block H prints them as Deck 3.5's two tables.
RESULTS = []
def record(table, algorithm, model, loss=None, loss_all=None, rmse=None, budget="", seconds=None):
    RESULTS.append(dict(table=table, algorithm=algorithm, model=model, loss=loss, loss_all=loss_all,
                        rmse=rmse, budget=budget, seconds=seconds))

# Block A — One model, one interface; the exact answer; the grade

*Deck 3.5: One Model, One Interface; Value Function Iteration and Policy Iteration.*

## A1. `reset`, `step`, `branches`, and the action mask

The text's running example has 7 capital levels and 2 productivity levels: 14 states. The action is next
period's capital, chosen from the same 7 values; only actions that leave consumption positive are allowed.
Two calls give one period: `branches` returns every $z'$ with its probability (the model in hand), `step` one
drawn $z'$ and the reward (all a learner without the model gets).

In [ ]:
env7 = s3.make_env_7x2(seed=0)
state = env7.reset(k_idx=3, z_idx=1)                       # k = 3.75, z = 1.1
print("state", state, "-> k =", env7.k_grid[state[0]], ", z =", env7.z_grid[state[1]])
print("feasible k':", env7.k_grid[env7.action_mask(state)])
for prob, nxt, r in s3.branches(env7, state, 3):                 # every branch of choosing k' = 3.75
    print(f"  branches: z' = {env7.z_grid[nxt[1]]} with probability {prob:.1f}, reward {r:.4f}")
next_state, reward, terminated, truncated, info = env7.step(3)     # choose k' = 3.75
print(f"chose k' = 3.75: reward = ln({info['consumption']:.4f}) = {reward:.4f}; next state {next_state}; "
      f"terminated = {terminated}")
masks = np.array([[env7.action_mask((k, z)).sum() for k in range(7)] for z in range(2)])
print("feasible actions per state (rows z = 0.9, 1.1):\n", masks, "-> total", masks.sum())
assert masks.sum() == 68 and abs(reward - math.log(1.3952880654722533)) < 1e-12

## A2. The exact solution: value function iteration

With the transition probabilities in hand, dynamic programming solves the model. `GrowthBenchmark` (book
ch. 12) does it to a tolerance of 1e-12 and keeps `V*`, `Q*` and the optimal rule `g*`. **No learner below is
ever given this object**; it is the answer key.

In [ ]:
bench7 = s3.GrowthBenchmark(env7)
trace = s3.vfi_trace(env7)
print(f"sweeps to 1e-6: {len(trace['change'])};  greedy rule equals g* from sweep "
      f"{int(np.argmax(trace['wrong_states'] == 0)) + 1}")
print("g*(k, z=0.9):", env7.k_grid[bench7.g_star[:7]]); print("g*(k, z=1.1):", env7.k_grid[bench7.g_star[7:]])
np.set_printoptions(suppress=True, linewidth=120)
Q = np.where(bench7.feasible, bench7.Q_star, np.nan)
print("\nQ*(s, a) for z = 0.9 (rows k, columns k'):"); print(np.round(Q[:7], 3))
top2 = np.sort(np.where(bench7.feasible, bench7.Q_star, -np.inf), axis=1)[:, -2:]
print(f"\nbest minus second best: median {np.median(top2[:, 1] - top2[:, 0]):.3f}, "
      f"smallest {np.min(top2[:, 1] - top2[:, 0]):.3f}")
assert len(trace["change"]) == 231 and abs(bench7.Q_star[3, 3] - 3.735) < 5e-4

fig, ax = plt.subplots(figsize=(5.2, 3.1))
ax.semilogy(np.arange(1, 232), trace["change"]); ax.set_xlabel("sweep"); ax.set_ylabel("max |V_n - V_(n-1)|")
ax.set_title("VFI on the 7x2 model: a contraction of modulus 0.95"); plt.tight_layout(); plt.show()

## A3. The grade: consumption-equivalent welfare loss

A learned rule `g` has an exact value `V^g` (solve a linear system). With log utility,

$$\ell_{CE}(s) = \exp\{(1-\gamma)\,[V^*(s) - V^g(s)]\} - 1$$

is the permanent percentage increase in consumption that would compensate a household for following `g` and not
`g*`. We report it averaged over all states, and weighted by how often the economy visits each state.
From here on the lab uses the book's **20 × 2 exercise grid** (40 states, 542 feasible pairs).

In [ ]:
env20 = s3.make_env_20x2(seed=0)
bench = s3.GrowthBenchmark(env20)
mu = bench.stationary_distribution(bench.epsilon_greedy_matrix(bench.g_star, 0.1))   # visit weights
loss_random = bench.ce_loss(bench.policy_value_stochastic(bench.uniform_random_matrix()))
loss_eps = bench.ce_loss(bench.policy_value_stochastic(bench.epsilon_greedy_matrix(bench.g_star, 0.1)))
print(f"states {bench.num_states}, feasible pairs {int(bench.feasible.sum())}")
print(f"uniform random rule      : loss {100 * loss_random.mean():6.2f}% (all states), {100 * mu @ loss_random:6.2f}% (weighted)")
print(f"g* with 10% random trials: loss {100 * loss_eps.mean():6.2f}% (all states), {100 * mu @ loss_eps:6.2f}% (weighted)")
# gate A: reproduce the two benchmark lines recorded in the book's README, to the printed digit
assert f"{100 * loss_random.mean():.2f}" == "107.08" and f"{100 * loss_eps.mean():.2f}" == "12.21"

def grade(g, label):
    m = bench.evaluate_greedy(np.asarray(g, dtype=int), mu)
    print(f"{label:34s} agrees with g* in {m['agree']:2d}/40 states; loss {100 * m['ce_all']:.3f}% (all), "
          f"{100 * m['ce_weighted']:.3f}% (weighted)")
    return m

## A4. Value function iteration and policy iteration on the 20 × 2 grid

Both read the model whole: every $z'$ summed with $P_z$. **VFI** backs up one period and takes the $\max$ every
sweep. **Policy iteration** (Howard) values the current rule all the way to convergence, $V = T^{g}V$ (one linear
solve), then improves it by the $\max$. Start from block D's rule, the median feasible $k'$. Few rounds, each
dearer; many sweeps, each cheap.

In [ ]:
t0 = time.time()
vfi20 = s3.vfi_trace(env20)
t_vfi = time.time() - t0
print(f"VFI: {len(vfi20['change'])} sweeps to 1e-6; greedy rule equals g* from sweep "
      f"{int(np.argmax(vfi20['wrong_states'] == 0)) + 1}  ({t_vfi:.2f} s)")
t0 = time.time()
pi_rounds = x.policy_iteration(bench, x.median_rule(bench), mu)
t_pi = time.time() - t0
for r in pi_rounds:
    print(f"PI round {r['round']}: rule agrees with g* in {r['agree']:2d}/40 states, loss {100 * r['ce_weighted']:7.3f}% "
          f"(weighted); improvement changes {r['changed']:2d} states")
print(f"({t_pi:.3f} s)")
assert pi_rounds[-1]["agree"] == 40 and pi_rounds[-1]["changed"] == 0          # gate: PI reaches g* exactly
record("with", "value function iteration", "20x2 grid", 0.0, 0.0, budget=f"{len(vfi20['change'])} sweeps", seconds=t_vfi)
record("with", "policy iteration", "20x2 grid", 100 * pi_rounds[-1]["ce_weighted"], 100 * pi_rounds[-1]["ce_all"],
       budget=f"{len(pi_rounds) - 1} improvements", seconds=t_pi)

## A5. The learner never reads the model

`ModelBlinded` passes `reset`, `step` and the action mask through and raises if anything asks for the
transition matrix, or for `branches`. Block E trains through it.

In [ ]:
blind = s3.ModelBlinded(s3.make_env_20x2(seed=1))
try:
    blind.p_z
except AttributeError as err:
    print("refused:", err)
try:
    s3.branches(blind, (10, 0), 10)
except AttributeError as err:
    print("refused:", err)
print("still works:", blind.reset(k_idx=10, z_idx=0), blind.step(10)[:2])

# Block B — The simplest actor–critic and its variants: eight numbers, with the model in hand

*Deck 3.5: The Simplest Actor–Critic; Its Variants: One Drawn Branch, and the Walk (the algorithm of Deck 3.4).*

Blocks A and D–F run on a grid. This block does not: it is Deck 3.4's demo, on the **continuous** growth model, where capital is a real number and the action is a saving share in $(0,1)$. The
learner is eight numbers. The **rule is deterministic**, as the growth model writes one:
$a=\mathrm{sigmoid}(\mu_\theta(k,z))$ with $\mu_\theta$ linear in $\ln k$ and the shock — three numbers. The critic is
$\widehat V_\psi=\psi^\top\phi$, five features. **The model is known.** Each iteration draws 256 states uniformly over
the band, forms the one-period backup $B(s)=u(c)+\gamma\sum_{z'}P_z(z'\mid z)\widehat V_\psi(k',z')$ with both shocks
weighted, refits the critic to $B$ by least squares, and moves the rule one gradient step up the Bellman right-hand
side, differentiating through the budget constraint. Nothing is simulated: the states are drawn, the lookahead is
summed. Before the first actor step the critic is prefitted to the frozen starting rule for 200 iterations.

The answer key is a 400-point dynamic programme with the maximum taken over a fine grid of shares. It is cached
in `session03_key.npz`; the first run rebuilds it in about half a minute.

## B1. The branches summed

**Reduced budget**: 5 seeds x 300 iterations, a few seconds. The slides quote 20 seeds x 1,000 iterations.

In [ ]:
import numpy as np, time
import session03_demo as demo                                  # course-owned; imports no book code

key = demo.cached_key()                                        # 400 k-points x 2 z, VFI to 1e-6
gr  = key.grid
print(f"steady state k* = {gr.k_star:.4f}   band [{np.exp(gr.lk_lo):.3f}, {np.exp(gr.lk_hi):.3f}]")
print(f"optimal share at k*: {key.share[2 * int(np.argmin(abs(gr.k - gr.k_star)))]:.4f}"
      f"   (the analytic steady-state share is {gr.k_star / (gr.k_star ** demo.ALPHA + (1 - demo.DELTA) * gr.k_star):.4f})")

In [ ]:
t0 = time.time()
settings = {k: v for k, v in demo.KNOWN.items() if k != "n_iter"}       # eta_theta, prefit, batch, features
run = demo.train_known(key, np.arange(5), n_iter=300, checkpoints=(10, 100), **settings)
share = demo.share_on_grid(gr, run.theta)
lived, band = demo.grade(gr, key, share)
crms, _ = demo.critic_rms(key, run.psi, share)
print(f"({time.time() - t0:.1f} s)  consumption-equivalent welfare loss of the learned rule, 5 seeds")
print(f"  where the economy lives : median {np.median(lived):.5f}%   worst seed {lived.max():.5f}%")
print(f"  over the whole band     : median {np.median(band):.5f}%")
print(f"  critic rms against V*   : median {np.median(crms):.3f} where the economy lives")
print(f"  slope theta_1           : median {np.median(run.theta[:, 1]):.3f}   (the best rule of this shape: 0.320)")
print(f"  prefit: critic error against the frozen rule's value fell from {run.prefit_err[0].mean():.2f} to {run.prefit_err[-1].mean():.3f}")
assert np.median(lived) < 0.005
record("with", "simplest actor-critic, branches summed", "continuous k", np.median(lived), np.median(band),
       budget="300 iterations x 256 states", seconds=time.time() - t0)

The graded object is the rule itself, $a=\mathrm{sigmoid}(\mu_\theta)$, and the weights come from where that rule's
own economy settles, computed by Young's two-point split, so the grade carries no simulation noise.

## B2. The variants: one drawn branch, the tree, the walk

The same eight numbers, the same budget; only the critic's target changes, one keyword each.

* `expectation="draw"`: one drawn $z'$ in the backup instead of the weighted pair. Unbiased, but noisy: the loss
  hovers rather than settles. This is the step from the summed backup to Deck 3.3's sampled one.
* `horizon=5, exact=True`: the target looks five periods ahead and enumerates all $2^5=32$ shock histories, **the tree**.
* `horizon=5, exact=False, paths=8`: the same five periods, sampled, **the walk**: eight runs of the rule per state.
  With `paths=1`, one run: a run ending on $\gamma^{5}\widehat V$ is $n$-step TD, and run to the end it would be
  Monte Carlo (block D).

Only the critic's target lengthens; the actor always climbs one period.

In [ ]:
variants = (("one drawn branch", dict(expectation="draw")),
            ("the tree, T = 5 (32 histories)", dict(horizon=5, exact=True)),
            ("the walk, T = 5, N = 8 runs", dict(horizon=5, exact=False, paths=8)),
            ("the walk, T = 5, N = 1 run", dict(horizon=5, exact=False, paths=1)))
results_B = {}
for label, kw in variants:
    t0 = time.time()
    rv = demo.train_known(key, np.arange(5), n_iter=300, **settings, **kw)
    lv, bd = demo.grade(gr, key, demo.share_on_grid(gr, rv.theta))
    results_B[label] = lv
    print(f"{label:32s} loss where the economy lives: median {np.median(lv):.5f}%, worst seed {lv.max():.5f}%  "
          f"({time.time() - t0:.1f} s)")
    record("with", label, "continuous k", np.median(lv), np.median(bd), budget="300 iterations x 256 states",
           seconds=time.time() - t0)
print(f"{'branches summed (B1)':32s} loss where the economy lives: median {np.median(lived):.5f}%, worst seed {lived.max():.5f}%")
assert np.median(results_B["one drawn branch"]) < 0.005
assert np.median(results_B["the tree, T = 5 (32 histories)"]) < 0.001
assert np.median(results_B["the walk, T = 5, N = 8 runs"]) < 0.003

Read the worst seed, not only the median: one drawn branch, or one run, costs little in the median and a great
deal in the worst case. That is the variance a sample buys in place of a sum. **Your turn**: try `horizon=10`
with `exact=True` and watch the time: $2^{10}$ leaves per state. Why does the walk not have that problem?

# Block C — Deep value iteration on Session 2's deterministic growth model

*Deck 3.5: Deep Value Iteration; Euler or Bellman: Two Roads on Session 2's Model.*

Session 2 solved this model on the Euler road with one network. Here are two: the **actor** is Session 2's
saving-share policy; the **critic** is a value network. Each step rolls the actor forward `n` periods through
the known law of motion and adds the critic's value of where it lands,

$$J_n(k) = \sum_{t<n}\beta^t u(c_t) + \beta^n V_\phi(k_n).$$

The actor climbs $J_n$ with the critic fixed; the critic regresses on $J_n$ with the target detached. It is
actor–critic with a known model: no sampling noise, no exploration — and still two moving targets.

**Calibration (Session 2, rung 0):** α = 0.33, β = 0.96, δ = 1, log utility, A = 1. Closed forms:
$k' = \alpha\beta k^\alpha$ and $V(k) = a + b\ln k$ with $b = \alpha/(1-\alpha\beta)$.

In [ ]:
g2 = value.g2
rung = g2.RUNGS[0]
ks = np.array([0.5, 1.0, 2.0]) * rung.k_ss
print("closed-form V at 0.5, 1, 2 x k*:", np.round(value.closed_form_value(rung, ks), 4),
      "| u(c*)/(1-beta) =", round(value.steady_state_value(rung), 4))

t0 = time.time()
bench_F = value.benchmarks(0)
actor, critic, history, seconds = value.train(value.CONFIGS[0])
v = value.validate(0, actor, critic, bench=bench_F)
gates = value.check_gates(v)
for name, ok in gates.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"policy error {100 * v['held_out_max_relative_policy_error']:.2f}%, value error "
      f"{100 * v['held_out_max_value_error_ce']:.3f}% of consumption, log10 Bellman residual "
      f"{v['held_out_log10_max_bellman_residual']:.2f}, log10 Euler error {v['held_out_log10_max_euler_error']:.2f}  ({seconds:.0f} s)")
record("with", f"deep value iteration, n = {value.CONFIGS[0].n_step}", "Session 2, rung 0",
       100 * v["held_out_max_value_error_ce"], budget=f"{value.CONFIGS[0].steps:,} steps x {value.CONFIGS[0].batch_size} states", seconds=seconds)
if not all(gates.values()):
    actor, critic, entry = value.load_reference(0)
    print("falling back to the hashed reference checkpoint:", entry["gates"])
    assert all(entry["gates"].values())

## C2. One step or many — and why a small residual is not a right answer

The same code with `n = 1` (pure one-step fitted value iteration). Compare the **Bellman residual** with the
**policy error** in the two runs.

In [ ]:
cfg1 = value.ValueConfig(**{**value.asdict(value.CONFIGS[0]), "n_step": 1})
actor1, critic1, _, sec1 = value.train(cfg1)
v1 = value.validate(0, actor1, critic1, bench=bench_F)
print(f"{'':28s}{'n = ' + str(value.CONFIGS[0].n_step):>12s}{'n = 1':>12s}")
for fld, label, scale in (("held_out_log10_max_bellman_residual", "log10 max Bellman residual", 1),
                          ("held_out_max_relative_policy_error", "max policy error (%)", 100),
                          ("held_out_max_value_error_ce", "max value error (% of c)", 100),
                          ("k_ss_relative_gap", "steady-state gap (%)", 100)):
    print(f"{label:28s}{scale * v[fld]:12.3f}{scale * v1[fld]:12.3f}")
record("with", "deep value iteration, n = 1", "Session 2, rung 0", 100 * v1["held_out_max_value_error_ce"],
       budget=f"{cfg1.steps:,} steps x {cfg1.batch_size} states", seconds=sec1)

k = torch.linspace(0.2, 2.0, 200).reshape(-1, 1) * rung.k_ss
with torch.no_grad():
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.1))
    ax[0].plot(k / rung.k_ss, bench_F["policy"](k.numpy().ravel()), "k--", label="closed form")
    ax[0].plot(k / rung.k_ss, actor(k, torch.ones_like(k)), label=f"n = {value.CONFIGS[0].n_step}")
    ax[0].plot(k / rung.k_ss, actor1(k, torch.ones_like(k)), label="n = 1")
    ax[0].set_title("policy k'"); ax[0].set_xlabel("k / k*"); ax[0].legend(frameon=False)
    ax[1].plot(k / rung.k_ss, bench_F["value"](k.numpy().ravel()), "k--"); ax[1].plot(k / rung.k_ss, critic(k))
    ax[1].plot(k / rung.k_ss, critic1(k)); ax[1].set_title("value V(k)"); ax[1].set_xlabel("k / k*")
plt.tight_layout(); plt.show()

With `n = 1` the critic is self-consistent with a *bad* actor: the pair satisfies its own Bellman equation
almost as well as the good pair does. The residual cannot tell them apart; the benchmark and the simulated steady
state can. This is Session 2's rung-1 lesson on the Bellman road, and the book's closing warning for
heterogeneous-agent models (ch. 12, "Knowing whether the answer is right").

## C3. Rung 1 from its reference checkpoint: Euler or Bellman

Rung 1 (CRRA 2, δ = 0.025, β = 0.99) takes about two minutes to train, so the lab loads the hashed reference
and reports its gates. Value-based training finds this rung hard for the reason Session 2 found:
with βR ≈ 1 the objective is nearly flat in the policy.

In [ ]:
actor_r1, critic_r1, entry = value.load_reference(1)
for name, ok in entry["gates"].items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
vv = entry["validation"]
print(f"rung 1: policy error {100 * vv['held_out_max_relative_policy_error']:.2f}%, steady-state gap "
      f"{100 * vv['k_ss_relative_gap']:.2f}%, log10 Euler error {vv['held_out_log10_max_euler_error']:.2f} "
      f"(trained in {entry['elapsed_seconds']:.0f} s)")
s2 = json.loads((LAB / "session02_assets" / "growth_ladder_report.json").read_text())["rungs"]["1"]["validation"]
print(f"Session 2, Euler road, same rung: policy error {100 * s2['held_out_max_relative_policy_error']:.2f}%, "
      f"steady-state gap {100 * s2['k_ss_relative_gap']:.2f}%")

# Block D — Monte Carlo: evaluate, then improve, from simulated paths

*Deck 3.5: Monte Carlo Evaluation; Monte Carlo Control. From here on we pretend not to know $P_z$.*

## D1. First-visit Monte Carlo evaluation of a fixed rule

The rule: always choose the median feasible `k'`. Its exact value is a linear solve; Monte Carlo estimates it
as an average of simulated returns. The error should fall like $1/\sqrt{N}$: sixteen times the episodes, a
quarter of the error.

In [ ]:
import mc_prediction as mcp                                   # book ch. 11
env = s3.make_default_env(seed=0)
rule = mcp.median_feasible_policy(env)
V_true = mcp.evaluate_policy_dp(env, mcp.policy_to_table(env, rule))
rmse = {}
for n in (250, 1000, 4000):
    V_hat, _ = mcp.first_visit_mc_prediction(s3.make_default_env(seed=0), rule,
                                             mcp.MCConfig(num_episodes=n, seed=0), snapshot_every=n)
    rmse[n] = float(np.sqrt(np.mean((V_hat - V_true) ** 2)))
    print(f"N = {n:5d} episodes: RMSE against the exact value = {rmse[n]:.4f}")
ratio = rmse[250] / rmse[4000]
print(f"error ratio for 16x the episodes: {ratio:.2f}  (1/sqrt(N) predicts 4)")
assert 2.5 < ratio < 7.5 and rmse[4000] < 0.07

## D2. Monte Carlo control: exploring starts against ε-greedy

Now the rule is improved as it is evaluated. Exploring starts begin each episode at a random state–action pair;
ε-greedy explores along the path, with $\varepsilon_n = \max(0.01, 1/\sqrt{n})$. Reduced budget: 6,000 episodes
(the recorded run uses 24,000). Monte Carlo is slow: 6,000 episodes are 600,000 periods of data, and block E
will do better with fewer.

In [ ]:
import mc_control as mcc                                      # book ch. 11
t0 = time.time()
results_D = {}
for name, fn in (("exploring starts", mcc.mc_control_es), ("epsilon-greedy", mcc.mc_control_eps_greedy)):
    t1 = time.time()
    _, g = fn(s3.make_default_env(seed=0), mcc.MCControlConfig(num_episodes=6000, seed=0))
    results_D[name] = m = grade(g, f"MC control, {name}")
    record("without", f"Monte Carlo control, {name}", "20x2 grid", 100 * m["ce_weighted"], 100 * m["ce_all"],
           budget="6,000 paths x 100 periods", seconds=time.time() - t1)
print(f"({time.time() - t0:.0f} s)")
assert results_D["exploring starts"]["ce_weighted"] < 0.12

The recorded full-budget run (`ch11/mc_control.py`, 24,000 episodes) is summarised below; the scripts print in
Chinese, so the notebook reads the numbers out of the recorded text. **Your turn** (book
§11.7.4, programming exercises): (b) is block D1 — now answer its second half: with paths cut at T = 100, how large
is the omitted tail, and how does it compare with the gaps between best and second-best actions in cell A2?
(d) the ε-greedy schedule in `mc_control_eps_greedy` is $\max(0.01, 1/\sqrt{n})$; change it to $1/n$, rerun, and
compare the final loss. Which schedule satisfies the conditions for convergence to the optimal rule, and which
does better at this sample size?

In [ ]:
print("\n".join(s3.recorded_digest(RECORDED, "ch11/mc_control.py")))

# Block E — Temporal difference: update every step

*Deck 3.5: TD(0); Between One Step and the Whole Path; SARSA; Q-Learning.*

## E1. Evaluate a rule: TD(0), n-step TD, Monte Carlo, TD(λ)

The rule: `g*` with 10 % random trials, the rule block F1 evaluates with five numbers. Here the value is a table,
learned through the model-blinded wrapper, with the step $\eta = C(k,z)^{-0.6}$. The target counts $n$ real
rewards and then trusts the table: $n=1$ is TD(0), $n=\infty$ is Monte Carlo (the whole path). TD(λ) averages
every $n$ with weights $(1-\lambda)\lambda^{n-1}$, online, through an eligibility trace. Graded by the error
against the exact $V^{\pi}$, weighted by where the economy lives. Three seeds, 100,000 transitions each, about
half a minute.

In [ ]:
pi_eval = bench.epsilon_greedy_matrix(bench.g_star, 0.1)
V_eval = bench.policy_value_stochastic(pi_eval)
dial = (("TD(0)", dict(n=1)), ("3-step TD", dict(n=3)), ("10-step TD", dict(n=10)),
        ("Monte Carlo (n = whole path)", dict(n=None)),
        ("TD(lambda = 0.5)", dict(lam=0.5)), ("TD(lambda = 0.9)", dict(lam=0.9)))
results_E1 = {}
for label, kw in dial:
    t0, errs = time.time(), []
    for sd in range(3):
        blind = s3.ModelBlinded(s3.make_env_20x2(seed=100 + sd))
        V_hat = x.td_prediction(blind, pi_eval, num_steps=100_000, seed=sd, **kw)
        assert blind.touched == [], "the learner tried to read the model"
        errs.append(x.weighted_rmse(V_hat, V_eval, mu))
    results_E1[label] = errs = np.array(errs)
    print(f"{label:30s} weighted RMSE: mean {errs.mean():.3f}, seeds {np.round(errs, 3)}  ({time.time() - t0:.1f} s)")
    record("evaluate", label, "20x2 grid", rmse=errs.mean(), budget="100,000 transitions", seconds=(time.time() - t0) / 3)
assert results_E1["TD(0)"].mean() < 0.12

ns = [1, 3, 10]
fig, ax = plt.subplots(figsize=(5.2, 3.1))
ax.plot(ns + [30], [results_E1[l].mean() for l in ("TD(0)", "3-step TD", "10-step TD", "Monte Carlo (n = whole path)")], "o-")
ax.set_xscale("log"); ax.set_xticks(ns + [30]); ax.set_xticklabels(["1", "3", "10", "whole\npath"])
ax.set_xlabel("real rewards in the target, n"); ax.set_ylabel("weighted RMSE of V")
ax.set_title("One dial from TD(0) to Monte Carlo"); plt.tight_layout(); plt.show()

At this budget the short target wins: each Monte Carlo return carries a hundred periods of shocks, and averaging
that noise away costs more data than TD's bias costs accuracy (that bias shrinks as the table improves). Larger
$n$ means less bias and more variance. **Your turn**: rerun with `num_steps=1_000_000`. Does the ranking survive?
Then try `step_exponent=1.0` (a plain running mean) for TD(0).

## E2. Control: SARSA, expected SARSA, Q-learning, double Q

Four learners on the same environment, through the model-blinded wrapper. 500,000 steps, one seed (the recorded
table uses 1,000,000 steps and three seeds). All explore with ε = 0.1 and step $\eta = C(s,a)^{-0.6}$; they differ
in one line, the value of the next state:

| learner | the target's next value | code |
|---|---|---|
| SARSA | $\widehat Q(s',a')$ at the $a'$ actually drawn | book |
| expected SARSA | the ε-greedy mean over feasible $a'$: same mean, less noise | `session03_extra` |
| Q-learning | $\max_{a'}\widehat Q(s',a')$ | book |
| double Q | two tables: one picks the $\arg\max$, the other prices it | `session03_extra` |

Expected SARSA's target has the same mean as SARSA's and less noise, so it converges to the same rule: the best
rule *given* 10 % random trials (the last line of the output). Whether the smaller noise shows at a finite budget is
an empirical question; here it does not: in the five-seed pilot expected SARSA's weighted loss is 0.9 %–2.4 %,
SARSA's 1.1 %–1.6 %. What holds SARSA back on this grid is its target rule, not the noise in its target.

In [ ]:
import tabular_td_control as tdc                              # book ch. 12
t0 = time.time()
results_E = {}
for method, label in (("sarsa", "SARSA"), ("expected_sarsa", "expected SARSA"), ("q", "Q-learning"), ("double_q", "double Q")):
    t1 = time.time()
    blind = s3.ModelBlinded(s3.make_env_20x2(seed=1000))
    cfg = tdc.TDControlConfig(num_steps=500_000, seed=0)
    q = (tdc.train_td_control(blind, method, cfg) if method in ("sarsa", "q")
         else x.train_td_control_x(blind, method, cfg))
    results_E[method] = m = grade(bench.greedy_from_q(q), f"{label}, 500,000 steps")
    assert blind.touched == [], "the learner tried to read the model"
    record("without", label, "20x2 grid", 100 * m["ce_weighted"], 100 * m["ce_all"], budget="500,000 transitions",
           seconds=time.time() - t1)
    if method == "double_q":
        i = np.arange(bench.num_states)
        print(f"   double Q's tables sit {np.mean(bench.Q_star[i, bench.g_star] - q[i, bench.g_star]):.2f} below Q* "
              f"at the optimal action, on average")
g_soft = tdc.epsilon_soft_optimum(bench, 0.1)
grade(g_soft, "best rule *given* 10% random trials")
print(f"({time.time() - t0:.0f} s)  the learners never touched: p_z, P")
assert results_E["q"]["ce_weighted"] < 0.02 and results_E["sarsa"]["ce_weighted"] < 0.025
assert results_E["expected_sarsa"]["ce_weighted"] < 0.035 and results_E["double_q"]["ce_weighted"] < 0.35

**Double Q is the slowest of the four here.** It removes the upward bias of the $\max$ in Q-learning's
target, but each table gets half the updates and is priced by the other, so at this budget the tables are still
far below $Q^*$ and the rule undersaves. The five-seed pilot: 1.6 %–24.9 % weighted loss, against 0.5 %–1.0 % for
Q-learning. A correction that pays off where the $\max$'s bias is large (many noisy actions with nearly equal
values) costs data where that bias is small. **Your turn**: give double Q `num_steps=2_000_000` and check whether
it catches up.

SARSA is not failing: it learns the best rule *for an agent who will keep making random trials*, and such an
agent should save a little less. The last line above is that rule's loss, computed exactly. **Your turn** (book
§12.8.4, programming exercise (b)): set ε = 0.3. First predict, with `tdc.epsilon_soft_optimum(bench, 0.3)`, in
which states SARSA's greedy choice will differ from `g*`; then check by sampling. **Recorded, full budget:**

In [ ]:
print("\n".join(s3.recorded_digest(RECORDED, "ch12/tabular_td_control.py")))

# Block F — Replace the table with parameters

*Deck 3.5: Linear TD and LSTD; DQN.*

## F1. Linear TD and LSTD: five numbers in place of forty

Evaluate the rule "g* with 10% random trials", block E1's rule, with the basis $\phi = (1, \tilde k, \tilde k^2, 1\{z_H\}, \tilde k\,1\{z_H\})$.
The best weighted least-squares fit, the TD fixed point, and LSTD from one sample path of 200,000 periods.
LSTD is an instrumental-variables estimator: instrument $\phi(s_t)$, regressor $\phi(s_t) - \gamma\phi(s_{t+1})$.

In [ ]:
import linear_td_approx as lin                                # book ch. 12
env = s3.make_env_20x2(seed=2024, episode_length=200_000)
pi = bench.epsilon_greedy_matrix(bench.g_star, 0.1)
V_pi, P_pi, r_pi = bench.policy_value_stochastic(pi), bench.transition_matrix(pi), bench.expected_reward(pi)
Phi, D = lin.feature_matrix(env), np.diag(mu)
psi_ls = np.linalg.solve(Phi.T @ D @ Phi, Phi.T @ D @ V_pi)
psi_td = np.linalg.solve(Phi.T @ D @ (Phi - bench.gamma * P_pi @ Phi), Phi.T @ D @ r_pi)
states, rewards = lin.simulate_path(env, bench.g_star, 0.1, 200_000, seed=7)
psi_lstd = lin.lstd(Phi, states, rewards, bench.gamma, 200_000)
err = {name: lin.weighted_rmse(Phi @ psi, V_pi, mu) for name, psi in
       (("best linear fit", psi_ls), ("TD fixed point", psi_td), ("LSTD, 200,000 periods", psi_lstd))}
for name, e in err.items():
    print(f"{name:24s} weighted RMSE {e:.4f}")
print(f"TD fixed point / best fit = {err['TD fixed point'] / err['best linear fit']:.2f}  (bound: 1/(1-gamma) = 20)")
for name, e in err.items():
    record("evaluate", f"linear: {name}", "20x2 grid", rmse=e, budget="200,000 periods" if "LSTD" in name else "exact")
assert err["LSTD, 200,000 periods"] < 0.3

## F2. A small DQN

A network maps the state to one value per action; replay and a target network keep the regression stable; the
action mask removes infeasible actions before every `max`. Reduced budget: 20,000 steps (recorded: 100,000).

In [ ]:
import dqn_growth as dqn                                      # book ch. 12
t0 = time.time()
env = s3.make_env_20x2(seed=0)
net = dqn.train_dqn(env, dqn.DQNConfig(total_steps=20_000, seed=0))
results_F = grade(dqn.greedy_policy(net, env), "DQN, 20,000 steps")
record("without", "DQN", "20x2 grid", 100 * results_F["ce_weighted"], 100 * results_F["ce_all"],
       budget="20,000 transitions", seconds=time.time() - t0)
print(f"({time.time() - t0:.0f} s)")
print("\n".join(s3.recorded_digest(RECORDED, "ch12/dqn_growth.py")))
assert results_F["ce_weighted"] < 0.035

# Block G — Adjust the policy directly

*Deck 3.5: REINFORCE; Actor–Critic Without the Model.*

## G1. On the grid: REINFORCE and one-step actor–critic

REINFORCE with a baseline (3,000 iterations) and one-step actor–critic (30,000 updates): the book's full budget,
about a minute and a half. The pilot showed why: at half the budget the neural actor–critic collapses on three
seeds of five. The policy is a masked softmax over feasible `k'`. Results are seed-sensitive — the book reports
three seeds, and so does the recorded output below.

In [ ]:
import policy_gradient as pg                                  # book ch. 12
t0 = time.time()
results_G = {}
t1 = time.time()
pol = pg.train_reinforce(pg.ReinforceConfig(seed=0))
results_G["reinforce"] = m = grade(pol.prob_table().argmax(axis=1), "REINFORCE + baseline, 3,000 iterations")
record("without", "REINFORCE + baseline", "20x2 grid", 100 * m["ce_weighted"], 100 * m["ce_all"],
       budget="3,000 iterations", seconds=time.time() - t1)
t1 = time.time()
pol = pg.train_actor_critic(pg.ActorCriticConfig(seed=0))
results_G["ac"] = m = grade(pol.prob_table().argmax(axis=1), "actor-critic, 30,000 updates")
record("without", "actor-critic", "20x2 grid", 100 * m["ce_weighted"], 100 * m["ce_all"],
       budget="30,000 updates", seconds=time.time() - t1)
print(f"({time.time() - t0:.0f} s)   random rule: {100 * mu @ loss_random:.1f}% weighted")
assert results_G["reinforce"]["ce_weighted"] < 0.035 and results_G["ac"]["ce_weighted"] < 0.02

In [ ]:
print("\n".join(s3.recorded_digest(RECORDED, "ch12/policy_gradient.py")))

**Your turn** (book §12.8.4, programming exercise (d)): drop the $\gamma^t$ weight from REINFORCE —
`pg.ReinforceConfig(seed=0, use_gamma_t=False)` — and compare the learned rule and its loss. (The run takes half a
minute; with seed 0 the loss rises to about 10 %, and the rule over-saves almost everywhere. Why?)

## G2. The eight numbers without the model

Block B's **eight numbers, now without the model**: one transition at a time from the simulator, a fixed
probe around the rule so that a different share is ever tried, and the score function for the gradient (Deck 3.4).
It needs a million transitions on the slides to reach 0.014 %; here 200,000, about 25 seconds.

In [ ]:
t0 = time.time()
run_mf = demo.train(key, np.arange(5),
                    eta_theta=demo.BASE["eta_theta"], eta_psi=demo.BASE["eta_psi"],
                    eta_lnsig=demo.BASE["eta_theta"] * demo.BASE["lnsig_ratio"],
                    learn_sigma=False, resets=True, n_steps=200_000)
lived_mf, band_mf = demo.grade(gr, key, demo.share_on_grid(gr, run_mf.theta))
print(f"({time.time() - t0:.0f} s)  model-free, 200,000 transitions, 5 seeds")
print(f"  where the economy lives : median {np.median(lived_mf):.4f}%   worst seed {lived_mf.max():.4f}%")
print(f"  over the whole band     : median {np.median(band_mf):.4f}%")
n_known = (settings["prefit"] + 300) * settings["batch"]           # prefit and training, every drawn state
print(f"  against the model in hand: {np.median(lived_mf) / max(np.median(lived), 1e-12):.0f} times the loss, "
      f"from {200_000 / n_known:.1f} times as many states ({n_known:,} drawn there, both branches summed at each)")
assert np.median(lived_mf) < 0.06
record("without", "actor-critic, the eight numbers", "continuous k", np.median(lived_mf), np.median(band_mf),
       budget="200,000 transitions", seconds=time.time() - t0)

The model-free learner never reads `p_z`: the transition matrix is used only inside the simulator, exactly as on the
grid. **Your turn**: pass `learn_sigma=True` to the model-free run, which makes the probe width a ninth parameter.
It gets *worse* — the slides record 0.035 % against 0.014 % at the full budget — because a learned $\sigma$ falls to
about 0.13 and the rule stops trying alternatives. Why should a parameter that is not part of the rule be allowed to
shrink, and what does it take with it when it does? And why does the learner with the model in hand need no probe at
all?

# Block H — Side by side

*Deck 3.5: With the Model, Side by Side; Without the Model, Side by Side.*

## H1. The deck's two tables, measured

Every row the blocks above recorded. **Loss** is the consumption-equivalent welfare loss in percent. For the grid
it is weighted by where the economy lives under `g*` with 10 % trials; for continuous $k$ it is weighted by where the
learned rule's own economy settles (median of five seeds). For deep value iteration it is the **largest** value
error over held-out states, in consumption. Compare rows within one model only: the three models have different
answer keys. The evaluation rows (TD prediction, linear TD) value a fixed rule and are graded by the weighted RMSE
of $V$ instead.

In [ ]:
def show(table, title, cols):
    rows = [r for r in RESULTS if r["table"] == table]
    print(title); print("-" * len(title))
    print(f"{'algorithm':42s}{'model':20s}" + "".join(f"{c:>12s}" for c, _ in cols) + f"  {'budget':30s}{'seconds':>8s}")
    for r in rows:
        vals = "".join(f"{r[k]:12.4f}" if r[k] is not None else f"{'':>12s}" for _, k in cols)
        sec = f"{r['seconds']:8.1f}" if r["seconds"] is not None else ""
        print(f"{r['algorithm'][:41]:42s}{r['model']:20s}{vals}  {r['budget'][:29]:30s}{sec}")
    print()

show("with", "WITH THE MODEL: every z' summed with P_z, or drawn from it",
     [("loss %", "loss"), ("loss % all", "loss_all")])
show("without", "WITHOUT THE MODEL: only (s, a, r, s') from step",
     [("loss %", "loss"), ("loss % all", "loss_all")])
show("evaluate", "EVALUATING A FIXED RULE (g* with 10% trials) without the model",
     [("RMSE of V", "rmse")])

Read the tables the way the deck does. **Down a column**, what changed: the sum became a draw, the table became a
function, the $\max$ became a step. **Along a row**, what one algorithm is, and what it cost. With the model, the
grid is solved exactly in a fraction of a second; without it, the best learners on the same grid are within a
fraction of a percent of consumption after hundreds of thousands of transitions. That gap is the price of not
knowing $P_z$. In Session 4 the law of motion of the distribution is usually truly unknown to the household; that
is where learning from `step` alone earns its place.

**Your turn**: pick the learner you would use on a model you cannot write down, and say which column of the table
decided it.

## H2. Continuous actions: a portfolio share

One risky asset, one safe asset, CRRA utility; the action is the risky share ω ∈ (0, 1). Two policies for a
bounded continuous action: a **Beta** distribution, and a **Gaussian squashed by a sigmoid**. The benchmark
share ω* ≈ 0.672 comes from Gauss–Hermite quadrature. Reduced budget: one seed each, 300 updates (recorded: 800
updates, three seeds).

In [ ]:
import portfolio_ac as pf                                     # book ch. 12
params = pf.PortfolioParams(); pbench = pf.MyopicBenchmark(params)
print(f"benchmark share omega* = {pbench.omega_star:.3f}")
t0 = time.time()
for name, cls in (("Beta", pf.BetaPolicy), ("sigmoid-Gaussian", pf.SigmoidGaussianPolicy)):
    env = pf.PortfolioEnv(params, seed=0)
    policy, _ = pf.train_continuous_pg(env, cls, pf.ContinuousPGConfig(num_iterations=300, seed=0))
    _, mean_omega, p10, p90 = pf.evaluate(env, policy, pbench, num_episodes=50)
    print(f"{name:18s} greedy share: mean {mean_omega:.3f} (10-90%: {p10:.3f}-{p90:.3f})")
print(f"({time.time() - t0:.0f} s)")
print()
print("\n".join(s3.recorded_digest(RECORDED, "ch12/portfolio_ac.py")))

# Notes to the decks — derivations and the chapters' reference material

The slides carry the statements; the arguments are here. Each note says which deck frames it supports and which
section of the book it follows (Miao & Feng, *AI for Economists*, ch. 11–12; English by the instructor). Nothing
below is executed.

## N1. When is the Bellman equation the same problem as the sequence problem?

*Supports Deck 3.1, "The Growth Model: a Sequence Problem, and Its Recursive Form" and "What Dynamic Programming Quietly Requires". Notation: these notes write the
policy as $\pi$ (the book writes $g$) and the discount factor as $\gamma\equiv\beta$.*

Deterministic growth model, return $F(k,k')=u(f(k)+(1-\delta)k-k')$, feasible set $\mathcal{A}(k)$, and
$V^{SP}(k_0)=\sup\lim_{T}\sum_{t\le T}\beta^tF(k_t,k_{t+1})$ over feasible sequences. Two assumptions: $\mathcal{A}(k)$
is non-empty, and the discounted sum has a limit (possibly infinite) on every feasible sequence. Then (Stokey,
Lucas and Prescott 1989, Theorems 4.2–4.5):

1. **$V^{SP}$ solves the functional equation.** Split any feasible plan into period 0 and the rest; the supremum of the rest is $V^{SP}(k_1)$.
2. **A solution $V$ with $\lim_t\beta^tV(k_t)=0$ on every feasible sequence equals $V^{SP}$.** Iterate the equation $n$ times; the limit condition kills the tail $\beta^nV(k_n)$.
3. **An optimal plan satisfies the equation period by period.** If a tail were not optimal from its own state, replacing it would raise the whole plan: the principle of optimality.
4. **A plan that satisfies it, with $\limsup_t\beta^tV^{SP}(k^*_t)\le0$, is optimal.** Iterate; the tail is non-positive in the limit.

Theorems 1–2 are about values, 3–4 about paths. Together: solving the Bellman equation and following its policy
gives the same value and the same path as solving the sequence problem. The two limit conditions rule out value
"pushed to infinity", as a transversality condition does. With bounded utility they hold automatically. **Log
utility is unbounded**, so the bounded-function argument does not apply as it stands; the standard repair is a
weighted sup norm or a growth condition on the value space (SLP ch. 9 for the stochastic case).

Why it matters here: every learner in this lab is graded against $V^*$; this is why $V^*$ is the right answer key.

*Miao & Feng, AI for Economists, ch. 11, "Theoretical foundations of the Bellman equation".*

## N2. Contraction, Blackwell's conditions, and the theorem of the maximum

*Supports Deck 3.1, "The Growth Model…" and "Two Value Functions: $V$ and $Q$"; Deck 3.3, "Why Tabular TD(0)
Converges: the Target Contracts".*

**Contraction mapping theorem.** On bounded continuous functions with the sup norm, the Bellman operator
$(T^*V)(k,z)=\max_{k'}\{u(c)+\beta E[V(k',z')\mid z]\}$ satisfies $\|T^*V_1-T^*V_2\|_\infty\le\beta\|V_1-V_2\|_\infty$.
By Banach's theorem it has a unique fixed point $V^*$, and $V_{n+1}=T^*V_n$ converges to it from any $V_0$, the
distance shrinking by at least $\beta$ per sweep.

**Blackwell's sufficient conditions** (1965). An operator $T$ is a contraction of modulus $\beta$ if it is
(i) *monotone*: $V_1\le V_2\Rightarrow TV_1\le TV_2$; (ii) *discounting*: $T(V+a)\le TV+\beta a$ for constants $a\ge0$.
Verification for $T^*$ in two lines: (i) $V$ enters the maximand with a positive sign, so a larger $V$ raises the
maximand at every $k'$ and hence its maximum; (ii) $E[(V+a)(k',z')]=E[V(k',z')]+a$, so the maximand rises by exactly
$\beta a$ at every $k'$. The same two lines work for the policy-evaluation operator $T^{\pi}$, which has no $\max$.

**Theorem of the maximum** (Berge). If the maximand (utility plus discounted continuation value) is jointly
continuous and the feasible correspondence is non-empty, compact-valued and continuous, then $V^*$ is continuous and
the optimal policy correspondence is upper hemicontinuous. With a convex feasible set and a strictly concave
maximand, the maximiser is unique and the policy is a continuous function. This is what licenses approximating the
policy by interpolation, or by a network.

Why it matters here: the contraction is the guarantee that sampling, bootstrapping and function approximation give
up piece by piece in Decks 3.2–3.4.

*Miao & Feng, AI for Economists, ch. 11, "Theoretical foundations of the Bellman equation".*

## N3. Howard's improvement step, modified policy iteration, and the stopping rule

*Supports Deck 3.1, "What Dynamic Programming Quietly Requires"; Deck 3.3, "The Backup
Spectrum".*

Between two maximisations, numerical practice often pushes the value forward several periods *under the current
policy*:
$$V_{n+1}(s)=E_{\pi}\Big[\sum_{t=0}^{T-1}\gamma^tr_{t+1}+\gamma^TV_n(s_T)\,\Big|\,s_0=s\Big].$$
$T=1$ is one step of policy evaluation. Larger $T$ puts $T$ periods of "true" reward in the update and shrinks the
weight on the old estimate to $\gamma^T$; evaluating the current policy more thoroughly before switching usually
converges faster. This is **modified policy iteration**. Two things should be kept apart: how *deep* to evaluate
(one step, several, to convergence) is the modified-policy-iteration question; Howard's (1960) *policy improvement
theorem* is a different statement, that the policy greedy in $Q^{\pi}$ is no worse than $\pi$.

On a table the $T$-step expectation is cheap: apply $T^{\pi}$ to the table $T$ times. The two costs are the table
itself, which grows exponentially in the state dimension, and, once no table can be kept, the tree of shock
histories from a single state ($2^{20}\approx10^6$ here; $7^{20}\approx8\times10^{16}$ with seven nodes). These are
the two sources of the curse of dimensionality; Monte Carlo attacks both.

**Stopping rule.** The loop stops when successive sweeps differ by less than tol. By the contraction property the
distance to the fixed point is then at most $\beta\,\text{tol}/(1-\beta)$: with $\beta=0.95$, nineteen times tol.

*Miao & Feng, AI for Economists, ch. 11, "Challenge one: the curse of dimensionality"; "Classical DP: full
expectation backups and multi-step unrolling"; "Value function iteration: how DP computes".*

## N4. An aside in continuous time: the Kolmogorov forward equation

*Supports Deck 3.1, "Challenge 3: an Unknown Law of Motion for the Distribution"; Deck 3.5, "Bridge (2): an Analytic Transition Matrix, Not a Simulated Panel". For readers who know continuous-time
heterogeneous-agent models; nothing else in the session depends on it.*

With density $\varphi_t(a,z)$, saving drift $\dot a(a,z)$ and switching intensities $\kappa$ between productivity
states,
$$\partial_t\varphi_t(a,z)=-\partial_a\big[\dot a(a,z)\varphi_t(a,z)\big]+\sum_{z'\ne z}\kappa_{z'z}\varphi_t(a,z')-\Big(\sum_{z'\ne z}\kappa_{zz'}\Big)\varphi_t(a,z).$$
The first term moves mass along the asset axis as households save or dissave; the other two are flows into and out
of productivity state $z$, a stock–flow relation. Together with the household's HJB equation this characterises the
model (Achdou et al. 2022). Writing the system down is not solving it: the saving rule depends on the value, the
value on the future distribution, the distribution on the saving rule — a coupled fixed point.

*Miao & Feng, AI for Economists, ch. 11, "Challenge two: incomplete knowledge of the environment" (think box).*

## N5. Importance sampling: the algebra, and why the horizon kills it

*Supports Deck 3.2, "Importance Sampling: Reweighting Someone Else's Data"; Deck 3.3, "Why Q-Learning Needs No
Importance Ratio".*

Data come from a behaviour policy $b$; we want the value of a target policy $\pi$. A trajectory from $s_t$ has
probability $\prod_j\pi(a_j\mid s_j)P(s_{j+1}\mid s_j,a_j)$ under $\pi$ and the same product with $b$ under $b$.
**The environment's factors are identical and cancel**, so the likelihood ratio needs no knowledge of $P$:
$$\rho_{t:T-1}=\prod_{j=t}^{T-1}\frac{\pi(a_j\mid s_j)}{b(a_j\mid s_j)}.$$
*Ordinary* importance sampling averages $\rho_{0:T-1}G_{0:T}$ over trajectories; *weighted* importance sampling
divides by the sum of the weights, not by the number of trajectories. Under **coverage** ($\pi(a\mid s)>0\Rightarrow
b(a\mid s)>0$) and finite episodes, the ordinary estimator is unbiased; its variance is finite only if the ratio has
a finite second moment. The weighted estimator is biased in finite samples and usually far less variable. For a
reader trained in causal inference these are the Horvitz–Thompson (inverse-probability-weighting) and Hájek
estimators; coverage is overlap.

**The variance problem.** The ratio is a product over periods. With a deterministic target, one deviation anywhere
sets the weight to zero; otherwise it is a product of $T$ numbers above one. In the book's example ($\epsilon=0.1$,
$T=50$) only 1.6% of trajectories keep a non-zero weight and the ordinary estimator's standard deviation is 1.303
against a target of 3.424; at $\epsilon=0.3$, $T=50$, no usable trajectory is left. Macro problems are long-horizon,
which is exactly where this fails. Remedies: truncated ratios, per-decision ratios, weighting, and, most important,
off-policy temporal-difference methods that never multiply along a path. Q-learning's one-step target
$r+\gamma\max_{a'}Q(s',a')$ points at the greedy action whatever policy generated the data.

*Miao & Feng, AI for Economists, ch. 11, "On-policy and off-policy methods".*

## N6. Three exploration mechanisms, and how fast exploration should fade

*Supports Deck 3.2, "Two Cures for Coverage: a Reset Button, or a Trembling Hand" and "Measured on the Whole Model: Greedy, Exploring Starts, $\epsilon$-Greedy";
Deck 3.3, "Which Cells Get Data: Explore Where It Can Pay".*

All mechanisms inject controlled randomness so that the visitation distribution does not collapse onto a few
state–action pairs. They differ in where the randomness enters:

| Mechanism | How it explores | How it exploits |
|---|---|---|
| exploring starts | randomises each trajectory's first pair $(s_0,a_0)$; needs a simulator that can be reset | greedy after the start |
| $\epsilon$-greedy | with probability $\epsilon$, a uniform draw from the feasible actions, every step | with probability $1-\epsilon$, the action currently estimated best |
| Boltzmann (softmax) | probabilities proportional to $\exp(\widehat Q(s,a)/\tau)$ | high estimates get high probability automatically |

At the book's reference state, $\epsilon=0.1$ costs 0.072 in expected value per decision (it tries the action that
makes consumption collapse as often as the near-best one); Boltzmann with $\tau=0.1$ costs 0.022. Too hot is costly
too: 0.141 at $\tau=0.3$, 0.406 at $\tau=1$.

**$\epsilon$-soft improvement** (Sutton and Barto 2018, §5.4). If $\pi$ is $\epsilon$-soft and $\pi'$ is
$\epsilon$-greedy in $Q^{\pi}$, then $V^{\pi'}\ge V^{\pi}$ everywhere; with equality, $\pi$ is optimal *among
$\epsilon$-soft policies*. So $\epsilon$-greedy control improves steadily — towards the best $\epsilon$-soft policy,
not towards $\pi^*$.

**GLIE** (greedy in the limit with infinite exploration): (1) every state–action pair is visited infinitely often;
(2) the policy becomes greedy in the limit, $\epsilon_n\to0$. Under GLIE, one-step on-policy methods such as SARSA
converge (Singh et al. 2000). $\epsilon_n=1/n$ and $1/\sqrt n$ both satisfy $\epsilon_n\to0$ and $\sum\epsilon_n=\infty$,
but the second is only necessary for infinite visits, and GLIE says nothing about finite samples: in the book's
experiment the $1/\sqrt n$ schedule ends at a median loss of 2.48%, worse than constant $\epsilon=0.1$ at 0.95%. A
floor $\epsilon_{\min}$ violates GLIE; what is learned is the best $\epsilon_{\min}$-soft policy, within
$O(\epsilon_{\min})$ of optimal.

*Miao & Feng, AI for Economists, ch. 11, "A unified view of exploration and exploitation" (incl. the GLIE think box).*

## N7. Chapter 11 in one table, and its check-list

*Supports Deck 3.1 as a whole, and the model-in-hand actor–critic of Deck 3.4 (the chapter's own summary).*

| Concept | The point |
|---|---|
| sequence problem and Bellman equation | under two assumptions and the limit conditions, four theorems make them the same problem (N1) |
| the MDP five-tuple | $(\mathcal{S},\mathcal{A},P,R,\gamma)$ maps one-to-one onto a dynamic economic model |
| Bellman equations | the expectation equation evaluates a policy; the optimality equation characterises the best one; $Q$ removes $P$ from the improvement step |
| full and sample backups | DP computes $\sum_{s'}P(s'\mid s,a)V(s')$ and needs the model and every state; Monte Carlo averages returns, with error $\sigma_G/\sqrt N$ whatever the dimension |
| generalised policy iteration | with exact evaluation greedy improvement is monotone; with estimates, precision problems fade with samples, coverage problems do not |
| on-policy and off-policy | the first learns the value of the exploring policy; the second needs importance ratios, whose variance explodes with the horizon |
| exploration and exploitation | data are generated by the policy; cells with zero visitation are not identified |

**From theory to practice:**

1. *Modelling.* Can the problem be written as an MDP? Name states, actions, reward and discount factor.
2. *Model knowledge.* Known law of motion and low dimension: value or policy iteration is more direct. Unknown transitions or high dimension: consider reinforcement learning.
3. *Simulator.* Even without explicit probabilities, can you generate $s'$ and $r$ from $(s,a)$? Then sample backups apply.
4. *Episodes.* Infinite-horizon problems must be truncated; check the discarded tail against the gaps between action values, or use TD.
5. *Exploration.* Can the simulator be reset? If so, exploring starts work well; if not, use $\epsilon$-greedy or Boltzmann, and report how many state–action pairs were actually covered.
6. *Equilibrium.* An MDP is one agent's problem. With prices set by many agents, add an outer loop between equilibrium conditions and policy learning.

*Miao & Feng, AI for Economists, ch. 11, "Key concepts at a glance"; "A check-list from theory to practice".*

## N8. The incremental mean, Robbins–Monro, and what a constant step costs

*Supports Deck 3.3, "The Sample Mean, Written One Observation at a Time" and "Target and Step Size Cannot Be Judged
Apart".*

The sample mean can be written one observation at a time,
$$\bar X_{n+1}=\bar X_n+\frac{1}{n+1}\big(X_{n+1}-\bar X_n\big):$$
new estimate = old estimate + step × (new observation − old estimate). Replace $1/(n+1)$ by a general step $\eta_n$.
The **Robbins–Monro conditions** $\sum_n\eta_n=\infty$, $\sum_n\eta_n^2<\infty$ guarantee convergence to the mean: the
first lets the steps travel from any start, the second keeps accumulated noise bounded. $\eta_n=1/n$ and
$\eta_n=n^{-\omega}$ with $1/2<\omega\le1$ both qualify.

A **constant step** $\eta$ fails the second condition. The estimate does not converge; it fluctuates around the mean
with stationary variance $\eta\sigma^2/(2-\eta)$, while the effect of the starting value decays like $(1-\eta)^n$.
What it buys is tracking: recent observations carry the weight, old ones fade geometrically. In reinforcement
learning the target usually *moves*, because the policy changes, so this property is used, not merely tolerated
(SARSA and Q-learning in Deck 3.3).

Why it matters here: with a bootstrapped target the step size and the target cannot be judged apart. The book's
experiment shows a $1/C(s)$ step, ideal for Monte Carlo, makes TD(0) very slow; $C(s)^{-0.6}$ with a sensible
starting value does far better.

*Miao & Feng, AI for Economists, ch. 12, "The incremental form of the sample mean".*

## N9. The TD fixed point: a projected Bellman equation, its error bounds, and LSTD

*Supports Deck 3.4, "Route One: What It Buys, What It Costs, and Why Route Two" (its guarantee row); lab block F1.
The deck states the result; the projected Bellman equation, the instrumental-variables reading and LSTD are here.*

Linear values $\widehat V_\psi=\Phi\psi$; semi-gradient TD updates $\psi\leftarrow\psi+\eta\,\delta_t\,\phi(s_t)$.
**Theorem** (Tsitsiklis and Van Roy 1997). If the chain under $\pi$ is irreducible and aperiodic with stationary
distribution $\bar d$, the data are one sample path under $\pi$, $\Phi$ has full column rank and the steps satisfy
Robbins–Monro, then $\psi$ converges with probability one to the unique solution of
$$\Phi^\top D(\Phi-\gamma P_\pi\Phi)\psi=\Phi^\top D\,r_\pi,\qquad D=\mathrm{diag}(\bar d).$$
This is $\Phi\psi=\Pi T^{\pi}\Phi\psi$ with $\Pi$ the $\bar d$-weighted least-squares projection: the **projected
Bellman equation**. It is *not* the best linear fit, but
$$\|\Phi\psi_{TD}-V^{\pi}\|_{\bar d}\le\frac{1}{1-\gamma}\min_\psi\|\Phi\psi-V^{\pi}\|_{\bar d}.$$

**The sharper bound.** Write $V^{\pi}-\Phi\psi_{TD}=(V^{\pi}-\Pi V^{\pi})+(\Pi V^{\pi}-\Phi\psi_{TD})$; the two pieces
are $\bar d$-orthogonal. The second equals $\Pi T^{\pi}V^{\pi}-\Pi T^{\pi}\Phi\psi_{TD}$; projection does not increase
the norm and $T^{\pi}$ is a $\gamma$-contraction in $\|\cdot\|_{\bar d}$, so its norm is at most
$\gamma\|V^{\pi}-\Phi\psi_{TD}\|_{\bar d}$. Pythagoras gives the factor $(1-\gamma^2)^{-1/2}$: about 3.2 at
$\gamma=0.95$, against 20. The condition that matters most is that the data come from the evaluated policy;
otherwise $D$ and $P_\pi$ no longer match and the iteration can diverge.

**LSTD as instrumental variables.** The fixed point is the moment condition
$E[\phi(s_t)(r_{t+1}+\gamma\phi(s_{t+1})^\top\psi-\phi(s_t)^\top\psi)]=0$. As a regression,
$r_{t+1}=(\phi(s_t)-\gamma\phi(s_{t+1}))^\top\psi+e_{t+1}$: the regressor contains $s_{t+1}$, whose randomness is also in
$e_{t+1}$, so OLS is biased. $\phi(s_t)$ is fixed before that randomness is realised — uncorrelated with the error,
correlated with the regressor: a valid instrument. Solving the sample moments gives
$\widehat\psi_{LSTD}=[\sum_t\phi(s_t)(\phi(s_t)-\gamma\phi(s_{t+1}))^\top]^{-1}\sum_t\phi(s_t)r_{t+1}$ (Bradtke and Barto
1996; Boyan 2002). It cannot use $\phi(s_{t+1})$ as the instrument, for the same reason OLS fails.

On the 20 × 2 grid of block F1 the book records: best fit 0.157, TD fixed point 0.236 (1.5 times, far inside the
bound of 20), LSTD on a million steps 0.234. It also records that the root-mean-square TD error is 0.57 *even with
the exact value function*: that is reward and transition noise, and must not be read as approximation error.

*Miao & Feng, AI for Economists, ch. 12, "Linear approximation and semi-gradient TD".*

## N10. The deadly triad: questions, and the principled remedies

*Supports Deck 3.4, "Route One: What It Buys, What It Costs, and Why Route Two" (its last row) and "Let the Features
Be Learned: a Network for $Q$".*

Tabular Q-learning converges even though it is off-policy and bootstraps, because each cell's error is corrected in
that cell alone. With a parameterised estimate three elements close a loop: **off-policy** data fall on a state
distribution that does not match the evaluated policy; **function approximation** makes one update move many
states; **bootstrapping** makes those moved predictions the next targets. An over-estimate raises its own target
and generalisation spreads it; every step is a reasonable gradient step and the whole can still blow up. Baird
(1995) built a diverging example with linear approximation; see Sutton and Barto (2018, §11.3).

Remedies with convergence guarantees in the linear off-policy case: **gradient-TD** methods (Sutton, Szepesvári and
Maei 2009; Sutton et al. 2009), which do stochastic gradient descent on the projected Bellman error itself, and
**emphatic TD** (Sutton, Mahmood and White 2016), which reweights updates with a follow-on trace to correct the
mismatch in visitation. A related point from the same section: taking the *full* gradient of $\tfrac12\delta_t^2$
gives the residual-gradient method, which minimises squared TD error; because the TD error contains next-period
noise, in a stochastic environment it converges to the wrong answer. "Semi-gradient" — in PyTorch, `detach()` on
the target — is a choice, not a shortcut.

**The book's questions.** (1) Evaluate the $\epsilon$-greedy rule with the five-term polynomial, but generate the
data with a uniformly random policy: which element of the triad was triggered? (2) Keep the data on-policy but
replace the linear basis by a network: what does the convergence theorem of N9 still guarantee? (3) DQN has all
three elements: on which one does each of its two devices act?

*Miao & Feng, AI for Economists, ch. 12, "Linear approximation and semi-gradient TD" (think box and footnotes).*

## N11. The policy-gradient theorem, with its proof, and what a baseline may depend on

*Supports Deck 3.4, "REINFORCE: Try, Score, Reinforce" and "REINFORCE with a Baseline".*

A trajectory $\tau$ has probability $P_\theta(\tau)=P_0(s_0)\prod_{t\ge0}\pi_\theta(a_t\mid s_t)P(s_{t+1}\mid s_t,a_t)$; only
the policy factors contain $\theta$. Objective: $J(\theta)=E[\sum_t\gamma^tr_{t+1}]$.

**Theorem** (Sutton et al. 2000; Marbach and Tsitsiklis 2001). With $\pi_\theta$ differentiable, bounded rewards and
$\gamma<1$,
$$\nabla_\theta J=E\Big[\sum_{t\ge0}\gamma^t\nabla_\theta\ln\pi_\theta(a_t\mid s_t)\,G_t\Big]=\frac{1}{1-\gamma}\sum_{s,a}d_{\pi_\theta}(s,a)\,\nabla_\theta\ln\pi_\theta(a\mid s)\,Q^{\pi_\theta}(s,a).$$

**Proof, in the book's three steps.** (1) Differentiate $J=\int P_\theta(\tau)R(\tau)d\tau$ and use
$\nabla P_\theta=P_\theta\nabla\ln P_\theta$: $\nabla J=E[\nabla\ln P_\theta(\tau)R(\tau)]$; bounded rewards let derivative
and integral be exchanged. (2) Take logs of $P_\theta(\tau)$: the initial distribution and the transition
probabilities do not contain $\theta$, so $\nabla\ln P_\theta(\tau)=\sum_t\nabla\ln\pi_\theta(a_t\mid s_t)$. *The
environment has dropped out.* (3) For $j<t$ the reward $r_{j+1}$ is determined before $a_t$ is drawn, and
$E_{a_t}[\nabla\ln\pi_\theta(a_t\mid s_t)]=\nabla\sum_a\pi_\theta(a\mid s_t)=0$, so those cross terms vanish; what remains
is $\sum_{j\ge t}\gamma^jr_{j+1}=\gamma^tG_t$. Conditioning $G_t$ on $(s_t,a_t)$ gives $Q^{\pi_\theta}$, and
$\sum_t\gamma^t\Pr(s_t=s,a_t=a)=d_{\pi_\theta}(s,a)/(1-\gamma)$ collects the periods.

**Keep the $\gamma^t$.** It is what weights each period's state distribution into the discounted visitation
distribution. Dropping it, as practice often does, gives a direction that is not the gradient of $J$ (Thomas 2014;
Nota and Thomas 2020). With separate parameters per state this only rescales steps; with shared parameters it
changes the objective. The book's network run without it learns to over-save almost everywhere and loses 10.2%.

**Baselines.** For any $v(s)$ that does not depend on the action,
$E_a[\nabla\ln\pi_\theta(a\mid s)\,v(s)]=v(s)\nabla\sum_a\pi_\theta(a\mid s)=0$, so subtracting $v(s_t)$ from $G_t$ leaves
the expectation unchanged and removes a zero-mean term highly correlated with the estimate: a control variate. With
$v=V^{\pi}$ the bracket's conditional mean is the advantage $A^{\pi}=Q^{\pi}-V^{\pi}$. $V^{\pi}$ is not the
variance-minimising baseline; that one weights $Q^{\pi}$ by the squared norm of the score (Greensmith, Bartlett and
Baxter 2004). The book's questions: does the identity survive a baseline that depends on $a_t$, and at which step
does the argument break? A baseline may depend on history before $t$ but on nothing realised after $a_t$ is drawn:
what is the condition behind that?

*Miao & Feng, AI for Economists, ch. 12, "The policy-gradient theorem"; "REINFORCE and baselines".*

## N12. Two time scales, and two ways to get the actor's gradient

*Supports Deck 3.4, "The Data Flow: One TD Error, Two Updates", "Deep Value Iteration: the Simplest Actor–Critic's Two Steps, with
Networks" and "Two Moving Targets, and a Residual That Can Mislead".*

**Why $\delta_t$ can replace $G_t-v(s_t)$.** If the critic is exact,
$E[\delta_t\mid s_t,a_t]=Q^{\pi}(s_t,a_t)-V^{\pi}(s_t)=A^{\pi}(s_t,a_t)$: the TD error is a one-step unbiased estimate of
the advantage. With an imperfect critic it is biased, and the actor is pulled the wrong way. That is the price of
updating every step, carrying the infinite tail, and having one period's noise where REINFORCE has a whole path's.

**Two time scales.** When the actor moves, the data distribution and the value move with it; a critic slower than
the actor reports a TD error for a policy that no longer exists. Practice gives the critic the larger step. Konda
and Tsitsiklis (2000) is the source of that design, but their theorem is narrow: finite states and actions, the
average-cost criterion, a *linear* critic whose features span the actor's score function (compatible features), and
an actor step that vanishes faster than the critic's. The algorithms of this session do not meet those conditions;
the citation explains where the design comes from, not that convergence is guaranteed. Implementation: in the
combined loss $\tfrac12\delta_t^2-\delta_t\ln\pi_\theta(a_t\mid s_t)$ treat the critic's target as a constant, and treat
$\delta_t$ as a constant in the actor's term.

**Score function or pathwise.** When the transition law is unknown, the actor's gradient must be *estimated* from
samples through the score $\nabla\ln\pi_\theta$ (Deck 3.4). When the model is known and differentiable — budget
constraint, prices, law of motion — the actor can differentiate the right-hand side of the Bellman equation
directly (Deck 3.4's deep value iteration, lab block C; Deck 3.5's Bridge (3), the book's heterogeneous-agent method). Same alternation of critic and actor; far
less variance.

*Miao & Feng, AI for Economists, ch. 12, "Actor–critic: the basic structure"; "Actor–critic for the household problem".*

## N13. The recorded results, method by method, and one further run

*Supports Deck 3.5, "With the Model, Side by Side" and "Without the Model, Side by Side". The deck walks each algorithm
through the growth model and prints no results; the book's recorded runs are here.*

Recorded by the book's script on the 7 × 2 grid (14 states, 20 seeds, $10^6$ steps; Monte Carlo $1.8\times10^6$): the
consumption-equivalent loss of the greedy rule each method learns, weighted by where the economy lives, and over all
states.

| method | estimates | loss: weighted / all |
|---|---|---|
| Monte Carlo, exploring starts | $Q^{g}$ | — / 0.20% |
| SARSA | the best $\epsilon$-soft rule's value | 0.18% / 0.38% |
| Q-learning | $Q^{*}$ | 0.00% / 0.01% |
| REINFORCE | the parameters $\theta$ | 1.68% / 2.10% |
| actor–critic | $\theta$ and $V^{g_\theta}$ | 0.53% / 1.30% |

REINFORCE and actor–critic run at actor step 0.5, with resets. This is not a ranking: with 14 states and few actions
Q-learning wins; actor–critic is for actions that cannot be listed, and for networks over large states.

One more row of the book's table is the actor–critic with a step that *decreases* with visits, $C(s)^{-0.8}$:

| actor–critic, decreasing step | $10^5$ steps | $3\times10^5$ | $10^6$ | seeds below 1% | random policy itself |
|---|---|---|---|---|---|
| median weighted loss | 2.29% | 2.70% | 2.69% | 8/20 | 5.45% |

It stalls at 2.69%, in the company of the too-small constant step (3.33%) and of the run without resets (2.11%).
The mechanism is the same in all three: the policy stays random for a long time, and what is best for an agent who
will keep acting randomly is to save less; by the time the policy concentrates, the economy no longer reaches the
high-capital states whose rule needs correcting. Its stochastic policy is also the most diffuse of the table
(5.45%). Recorded by the book's script on the 7 × 2 grid (20 seeds); the full output is in
`book_recorded_results.json` under `ch12/running_example.py`.

**A further run.** `ch12/policy_gradient.py --ppo` adds a simplified PPO-clip to the network versions of REINFORCE
and actor–critic on the 20 × 2 grid (about two and a half minutes in all). The lab does not run it and no slide
quotes it. PPO's clipped objective, the one-step importance ratio inside it and generalised advantage estimation
are on no slide: this run is the only place in Session 3 that touches them.

*Miao & Feng, AI for Economists, ch. 12, "Solving the growth model: where the chapter's methods meet".*

## N14. Platforms, data-governance rules as constraints on the action set, and one question

*Supports Deck 3.5, "When Dynamic Programming, When Reinforcement Learning".*

The slide's fourth application, in the book's fuller form. In dynamic pricing, ad placement and mechanism design a
firm's action changes future demand and the competitive landscape, so the environment contains other participants'
strategic responses: closer to *learning in equilibrium* than to a single-agent MDP. The book points to deployments
at industrial scale on Chinese internet platforms: real-time bidding for display advertising and ranking in search
at Alibaba (Cai et al. 2017; Chen et al. 2019), and order dispatch and dynamic scheduling across cities at DiDi
with large-scale distributed reinforcement learning (Xu et al. 2018). In these systems the platform is better
described as a mechanism designer learning in equilibrium, which makes concrete the boundary between "one agent"
and "equilibrium" drawn in Deck 3.1.

These deployments operate under China's Personal Information Protection Law and Data Security Law (both 2021). The
book's observation is an economist's one: rules on cross-border data flows, on re-using user-level features, and on
differentiated treatment of minors enter the algorithm as **restrictions on the feasible action set**
$\mathcal{A}(s)$ — the same object as the consumption-positivity mask of block A.

**The book's question** (think box, "From economics to modern AI"). AlphaGo can be read as: approximate value and
policy with networks in a huge state space, and solve exploration and intertemporal credit assignment by self-play.
Economic problems have different structure and constraints, but the core logic — MDP, Bellman structure, learning
from samples — is the same. *In a dynamic economic model you know well, which part is hardest to learn by trial and
error in the real world, and therefore most needs a calibrated simulator?*

*Miao & Feng, AI for Economists, ch. 12, "Applications in economics".*

## N15. Representing a distribution: six routes, and two implementation details

*Supports Deck 3.5, the three "Bridge" frames. The book is explicit, and so is this note: for the last route this is
**method only**. In the book's words, "this research is still in progress; this section presents the method and
reports no numerical results" (Feng, Han, Sargent and Zhu 2026). Session 4 takes these routes up.*

| Approach | How the distribution is represented | Strength and cost | References |
|---|---|---|---|
| global solution with a few moments | aggregate capital (more moments can be added) and a forecasting rule | fast, accurate in standard models; distorts when the *shape* matters; the rule must be re-estimated when policy changes | Krusell and Smith (1998); Den Haan (2010); Young (2010) |
| perturbation around the steady state | a discretised distribution, linearised around the steady state without aggregate shocks | handles very large distribution states quickly; accurate for small shocks only | Reiter (2009) |
| sequence-space Jacobian | linearisation around the steady state in the space of shock paths | extremely efficient for HANK models with small shocks; distorts under strong non-linearity and large shocks | Auclert et al. (2021) |
| continuous-time methods | the forward equation for the density, discretised on a grid | fast for stationary equilibria and transitions; with aggregate shocks usually combined with perturbation | Achdou et al. (2022) |
| global solution with a histogram | a histogram vector, moved by an analytic transition matrix | no shape assumed, the whole distribution kept; state dimension equals the number of bins, training is costly | Azinovic et al. (2022); Feng, Han, Sargent and Zhu (2026) |
| simulated panel with learned moments | a few "generalised moments" learned by a network | low dimension, invariant to permuting households; the moments have no transparent economic meaning | Han, Yang and E (2026) |

**Two details of the household actor–critic.** (1) *Where the actor's derivative comes from.* When $\gamma(1+r)$ is
close to one, the two terms of the first-order condition of the Bellman right-hand side nearly cancel and the
value network's derivative is too noisy to steer by. The implementation takes the *level* from the value network
but replaces the derivative with respect to $a'$ by the Euler-equation expression $E[(1+r')u'(c')]$, and adds the
residuals of the Euler equation and its complementary-slackness condition as an auxiliary penalty, for accuracy near
the borrowing limit. (2) *What the saving network outputs.* Not the level of next assets but the *change*:
$a'=\min\{\max\{a+D(2o-1),\underline a\},\bar a\}$ with $o\in(0,1)$ from a sigmoid and $D$ a cap on the per-period
change. Near the stationary equilibrium average changes are small and small systematic errors in the saving rule
are magnified in the stationary distribution; the change form represents "no change" exactly and lets network error
enter at the scale of $D$, not of the whole asset range. It is the same move as block C's saving share and block A's
mask: write known economic structure into the parameterisation.

*Miao & Feng, AI for Economists, ch. 12, "A finite-dimensional representation: the histogram"; "Actor–critic for
the household problem".*

## N16. Chapter 12 in one table, and its seven-item check-list

*Supports Decks 3.2–3.4 as a whole (the chapter's own summary).*

| Concept | The point |
|---|---|
| incremental mean and step size | new = old + step × (observation − old); Robbins–Monro gives convergence, a constant step tracks a moving target |
| TD target and bootstrapping | the sample backup with $T=1$; biased, but its conditional mean is $T^{\pi}\widehat V$; converges in the tabular case |
| TD against Monte Carlo | MC: no bootstrap bias, high variance; TD: low variance, slow to unlearn a bad start; avoid $1/C$ steps for TD |
| $n$-step returns and TD($\lambda$) | a continuum between one step and the whole path; eligibility traces implement the $\lambda$ weights online |
| SARSA and Q-learning | next value at the action taken, or at the best action; the first learns the best $\epsilon$-soft rule (under-saving here), the second $Q^*$, needing coverage and over-estimating |
| regression replaces interpolation | samples sit on the visitation distribution; linear TD solves the projected Bellman equation; LSTD is its instrumental-variables estimator |
| deadly triad and DQN | off-policy + approximation + bootstrapping can diverge; a target network freezes the label, replay breaks serial correlation |
| policy-gradient theorem and baselines | the environment's factors vanish from the gradient; a baseline is a control variate |
| the cost of REINFORCE | whole returns in place of $Q^{\pi_\theta}$: wait for the path, truncate the tail, carry its variance |
| actor–critic | a TD-trained critic carries the tail; $\delta_t$ estimates the advantage; without exploration and a suitable step it locks in |
| score function and pathwise gradient | unknown model: estimate the gradient from samples; known differentiable model: differentiate through it |
| the distribution's transition matrix | histogram plus two-point split: mass conserved, differentiable in the policy; households hold it fixed, a government differentiates through it |

**From theory to practice:**

1. *Benchmark first.* Solve a small version exactly; convert every learned rule into a consumption-equivalent loss.
2. *Truncation is not termination.* Keep the bootstrap term at a truncation or reset; drop it only at a true terminal state.
3. *Step sizes and starting values.* For TD avoid $1/C$; use a constant or $C^{-\omega}$ with $1/2<\omega<1$, and an economically sensible start.
4. *Exploration and coverage.* Report how many pairs were adequately sampled, and both weighted and simple-average losses; if the simulator can be reset, reset it.
5. *Whose value?* SARSA's is the exploring policy's, Q-learning's the ideal policy's. Say which one a welfare number refers to.
6. *Seeds and uncertainty.* Repeat over seeds; report medians and quantiles, not the best run (Henderson et al. 2018).
7. *Is deep RL needed?* High dimension or continuous actions? A small benchmark available? Budget for many seeds? For equilibrium models, judge a solution at the horizon that determines the object of interest, not by a one-step residual.

*Miao & Feng, AI for Economists, ch. 12, "Key concepts at a glance"; "A check-list from theory to practice".*

## N17. The chapters' exercises

*Supports the "your turn" prompts of blocks D and G, and leaves the rest with the reader.*

**Chapter 11 (§11.7.4).**

- *Concepts* (a)–(d): full against sample backups and what each needs; the one formal difference between the expectation and optimality equations; why $Q$ beats $V$ when $P$ is unknown; why a deterministic optimum still needs a random learner, separating precision from coverage.
- *Derivations* (a)–(c): derive the Bellman expectation equation from $G_t=r_{t+1}+\gamma G_{t+1}$; prove $T^{\pi}$ is a $\gamma$-contraction by Blackwell's conditions (N2); a two-state, two-action MDP solved by hand, with one step of greedy improvement.
- *Programming* (a)–(e): (a) build the 20 × 2 environment with `reset`/`step` and solve it by VFI — this is block A; (b) first-visit Monte Carlo evaluation and its convergence curve — block D1, whose prompt builds on it; (c) exploring-starts control, graded in consumption units — block D2; (d) $\epsilon$-greedy control under two decay schedules and the GLIE trade-off — block D2's second prompt is a variation on it; (e) reproduce the coverage experiment of the text with `ch11/running_example.py`, changing how often cells are re-checked. **Item (e) is not in the lab**; the script is vendored and runs as it stands.
- *Applications* (a)–(d): Rust's bus-engine problem as an MDP; why a heterogeneous-agent model is not a single-agent MDP; the size of the state once a histogram enters, and when moment truncation fails; portfolio choice with a continuous action (the lab's last block).

**Chapter 12 (§12.8.4).**

- *Concepts* (a)–(d): why $1/C(s)$ suits Monte Carlo and cripples TD(0); why SARSA under-saves, and what that means for evaluating rules with execution noise; the triad with a five-term basis; what REINFORCE brought back and how the critic removes it, including the role of resets.
- *Derivations* (a)–(d): $E[\delta_t\mid s_t,a_t]=A^{\pi}$ with an exact critic, and the bias otherwise; the baseline lemma and a counter-example with an action-dependent baseline (N11); LSTD from its moment condition, naming instrument and endogenous regressor (N9); the sample backup at $T=1$, $n$, $\infty$, and the recursion for the $\lambda$-return.
- *Programming* (a)–(f): (a) TD(0) step-size exponents; (b) SARSA at $\epsilon=0.3$, predicted exactly then sampled — block E's prompt; (c) a basis of your own for linear TD — a natural extension of block F1; (d) REINFORCE without $\gamma^t$, and with shorter paths — block G's prompt; (e) a 7-bin transition matrix by the two-point split, against a simulated panel; (f) Beta against sigmoid-Gaussian policies with `ch12/portfolio_ac.py` — the lab's last block. Items (a) and (e) are left with the reader.
- *Applications* (a)–(b): evaluating a monetary rule from historical data (estimating the behaviour policy, coverage, unobserved confounding); moments against histograms in Krusell–Smith, and why the analytic transition matrix cannot be replaced in the government's problem.

*Miao & Feng, AI for Economists, ch. 11, "Exercises"; ch. 12, "Exercises".*

## N18. Further reading, as the chapters give it

**Chapter 11 (§11.7.5).**

- Sutton and Barto, *Reinforcement Learning: An Introduction*, 2nd ed. (2018): the chapter's main reference; chs. 2–6.
- Zhao, *Mathematical Foundations of Reinforcement Learning* (Springer, 2025): the mathematically rigorous companion.
- Stokey, Lucas and Prescott, *Recursive Methods in Economic Dynamics* (1989), chs. 4 and 9; Ljungqvist and Sargent, *Recursive Macroeconomic Theory* (2012).
- Bertsekas and Tsitsiklis, *Neuro-Dynamic Programming* (1996): error bounds for approximate policy iteration.
- Maliar, Maliar and Winant (2021): deep learning for high-dimensional dynamic models.
- Judd, *Numerical Methods in Economics* (1998): projection methods, sparse grids.
- Qiu Xipeng, *Neural Networks and Deep Learning* (2020), ch. 14 (in Chinese); OpenAI, *Spinning Up in Deep RL*, "Key Concepts in RL".

**Chapter 12 (§12.8.5).**

- Sutton and Barto (2018), chs. 6–7, 9–11, 12, 13; Zhao (2025), chs. 6–10, section by section with the chapter.
- TD and function approximation: Tsitsiklis and Van Roy (1997); Bradtke and Barto (1996); Boyan (2002).
- Origins: Watkins and Dayan (1992), Q-learning; Williams (1992), REINFORCE; Sutton et al. (2000), the policy-gradient theorem; Konda and Tsitsiklis (2000), two time scales; Greensmith, Bartlett and Baxter (2004), baselines.
- Deep reinforcement learning: Mnih et al. (2015), DQN; van Hasselt, Guez and Silver (2016), Double DQN; Schulman et al. (2017), PPO; Lillicrap et al. (2015), DDPG; Haarnoja et al. (2018), SAC; Henderson et al. (2018), reproducibility.
- Deep learning for heterogeneous-agent models: Maliar, Maliar and Winant (2021); Azinovic, Gaegauf and Scheidegger (2022); Han, Yang and E (2026); Feng, Han, Sargent and Zhu (2026); Feng (2025), PyTorch implementations with code.

*Miao & Feng, AI for Economists, ch. 11, "Further reading"; ch. 12, "Further reading".*

# What to take away

1. One Bellman backbone; the methods differ in how the expectation is filled (full, sampled, bootstrapped) and
   in what stores the value (a table, five coefficients, a network).
2. Every method was graded against an exact answer, in units of consumption — not by its own loss.
3. A small residual is evidence, not a right answer: block C2.
4. Deep methods earn their place in dimension, not here. Session 4 puts a wealth distribution in the state.